# Text-Image to Video generation with Wan2.2 and OpenVINO

 Wan2.2 is a comprehensive and open suite of video foundation models that pushes the boundaries of video generation. Wan2.2 is a major upgrade to Wan2.1 which includes following features:

- **Effective MoE Architecture**: Wan2.2 introduces a Mixture-of-Experts (MoE) architecture into video diffusion models. By separating the denoising process cross timesteps with specialized powerful expert models, this enlarges the overall model capacity while maintaining the same computational cost.

- **Cinematic-level Aesthetics**: Wan2.2 incorporates meticulously curated aesthetic data, complete with detailed labels for lighting, composition, contrast, color tone, and more. This allows for more precise and controllable cinematic style generation, facilitating the creation of videos with customizable aesthetic preferences.

- **Complex Motion Generation**: Compared to Wan2.1, Wan2.2 is trained on a significantly larger data, with +65.6% more images and +83.2% more videos. This expansion notably enhances the model's generalization across multiple dimensions such as motions, semantics, and aesthetics, achieving TOP performance among all open-sourced and closed-sourced models.

- **Efficient High-Definition Hybrid TI2V**: Wan2.2 open-sources a 5B model built with our advanced Wan2.2-VAE that achieves a compression ratio of 16×16×4. This model supports both text-to-video and image-to-video generation at 720P resolution with 24fps and can also run on consumer-grade graphics cards like 4090. It is one of the fastest 720P@24fps models currently available, capable of serving both the industrial and academic sectors simultaneously.

You can find more details about model in [model card](https://huggingface.co/Wan-AI/Wan2.2-TI2V-5B-Diffusers) and [original repository](https://github.com/Wan-Video/Wan2.2)

In this tutorial we consider how to convert, optimize and run Wan2.2 model for Text-Image to Video generation using OpenVINO.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Convert model to OpenVINO Intermediate Representation](#Convert-model-to-OpenVINO-Intermediate-Representation)
    - [Compress model weights](#Compress-model-weights)
- [Prepare model inference pipeline](#Prepare-model-inference-pipeline)
    - [Select inference device](#Select-inference-device)
- [Run OpenVINO Model Inference](#Run-OpenVINO-Model-Inference)
- [Interactive demo](#Interactive-demo)


### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO. It may be fully supported and validated in the future.

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/wan2.2-text-image-to-video/wan2.2-text-image-to-video.ipynb" />


## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
import platform

%pip install -q "torch>=2.1" "diffusers==0.35" "transformers==4.49.0" "accelerate" "safetensors" "sentencepiece" "peft>=0.15.0" "ftfy" "gradio>=4.19" "opencv-python" "imageio[ffmpeg]" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install --pre -U "openvino>=2025.4.0" "nncf>=2.16.0" --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly

if platform.system() == "Darwin":
    %pip install -q "numpy<2.0.0"

Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://pypi.org/simple, https://storage.openvinotoolkit.org/simple/wheels/nightly
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.4/53.4 MB 17.1 MB/s eta 0:00:00m eta 0:00:010:01:01
  Using cached nncf-2.19.0-py3-none-any.whl (1.5 MB)
  Using cached openvino_telemetry-2025.2.0-py3-none-any.whl (25 kB)
  Using cached natsort-8.4.0-py3-none-any.whl (38 kB)
  Using cached pymoo-0.6.1.6-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (5.0 MB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
  Using cached scikit_learn-1.7.2-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (9.7 MB)
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (180 kB)
  Using cached pydot-3.0.4-py3-none-any.whl (35 kB)
  Using cached tabulate-0.9.0-py3-none-any.whl (35 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl (122 kB)
 

In [2]:
import requests
from pathlib import Path

if not Path("ov_wan_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/notebooks/wan2.2-text-image-to-video/ov_wan_i2v_helper.py")
    open("ov_wan_helper.py", "w").write(r.text)

if not Path("gradio_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/notebooks/wan2.2-text-image-to-video/gradio_helper.py")
    open("gradio_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

## Convert model to OpenVINO Intermediate Representation
[back to top ⬆️](#Table-of-contents:)


Wan2.2 is PyTorch model. OpenVINO supports PyTorch models via conversion to OpenVINO Intermediate Representation (IR). [OpenVINO model conversion API](https://docs.openvino.ai/2024/openvino-workflow/model-preparation.html#convert-a-model-with-python-convert-model) should be used for these purposes. `ov.convert_model` function accepts original PyTorch model instance and example input for tracing and returns `ov.Model` representing this model in OpenVINO framework. Converted model can be used for saving on disk using `ov.save_model` function or directly loading on device using `core.compile_model`.

Model consist of 4 parts:
* **Text Encoder** to encode input multi-language text, incorporating cross-attention within each transformer block to embed the text into the model structure
* **Diffusion Transformer** for step by step denoising of generated video guided by text instructions.
* **VAE Encoder** to encode generated video to latent space representation.
* **VAE Decoder** to decode generated video from latent space representation.

Model performs text-to-video generation task. For preserving original model flexibility, we will convert each part separately.

The script `ov_wan_i2v_helper.py` contains helper function for model conversion, please check its content if you interested in conversion details.

### Compress model weights
[back to top ⬆️](#Table-of-contents:)

For reducing memory consumption, weights compression optimization can be applied using [NNCF](https://github.com/openvinotoolkit/nncf). 

<details>
    <summary><b>Click here for more details about weight compression</b></summary>
Weight compression aims to reduce the memory footprint of a model. It can also lead to significant performance improvement for large memory-bound models, such as Large Language Models (LLMs). LLMs and other models, which require extensive memory to store the weights during inference, can benefit from weight compression in the following ways:

* enabling the inference of exceptionally large models that cannot be accommodated in the memory of the device;

* improving the inference performance of the models by reducing the latency of the memory access when computing the operations with weights, for example, Linear layers.

[Neural Network Compression Framework (NNCF)](https://github.com/openvinotoolkit/nncf) provides 4-bit / 8-bit mixed weight quantization as a compression method primarily designed to optimize LLMs. The main difference between weights compression and full model quantization (post-training quantization) is that activations remain floating-point in the case of weights compression which leads to a better accuracy. Weight compression for LLMs provides a solid inference performance improvement which is on par with the performance of the full model quantization. In addition, weight compression is data-free and does not require a calibration dataset, making it easy to use.

`nncf.compress_weights` function can be used for performing weights compression. The function accepts an OpenVINO model and other compression parameters. Compared to INT8 compression, INT4 compression improves performance even more, but introduces a minor drop in prediction quality.

More details about weights compression, can be found in [OpenVINO documentation](https://docs.openvino.ai/2024/openvino-workflow/model-optimization-guide/weight-compression.html).
</details>

In [3]:
import ipywidgets as widgets

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("wan2.2-text-image-to-video.ipynb")

model_id = "Wan-AI/Wan2.2-TI2V-5B-Diffusers"
model_base_dir = Path(model_id.split("/")[-1])

model_format = widgets.Dropdown(
    options=["FP16", "INT8", "INT4"],
    value="INT4",
    description="Model format:",
)

model_format

Dropdown(description='Model format:', index=2, options=('FP16', 'INT8', 'INT4'), value='INT4')

In [4]:
import nncf

model_dir = model_base_dir / model_format.value

if model_format.value == "INT4":
    weights_compression_config = {"mode": nncf.CompressWeightsMode.INT4_ASYM, "group_size": 64, "ratio": 1.0}
elif model_format.value == "INT8":
    weights_compression_config = {"mode": nncf.CompressWeightsMode.INT8_ASYM}
else:
    weights_compression_config = None

In [5]:
from ov_wan_i2v_helper import convert_pipeline

# Uncomment the line to see model conversion code
# ??convert_pipeline

In [ ]:
convert_pipeline(model_id, model_dir, compression_config=weights_compression_config)

## Prepare model inference pipeline
[back to top ⬆️](#Table-of-contents:)


`OVWanImageToVideoPipeline` defined in `ov_wan_i2v_helper.py` provides unified interface for running model inference. It accepts model directory and target device map for inference.

In [7]:
from ov_wan_i2v_helper import OVWanImageToVideoPipeline

# Uncomment the line to see model inference code
# ??OVWanImageToVideoPipeline

### Select inference device
[back to top ⬆️](#Table-of-contents:)

You can specify inference device for each pipeline component or use the same device for all of them using widgets below.

In [8]:
from notebook_utils import device_widget

device_transformer = device_widget(exclude=["NPU"], description="Transformer")
device_text_encoder = device_widget(exclude=["NPU"], description="Text Encoder")
device_vae_decoder = device_widget(exclude=["NPU"], description="VAE Decoder")
device_vae_encoder = device_widget(exclude=["NPU"], description="VAE Encoder")


devices = widgets.VBox([device_transformer, device_text_encoder, device_vae_decoder, device_vae_encoder])

devices

In [11]:
device_map = {
    "transformer": device_transformer.value,
    "text_encoder": device_text_encoder.value,
    "vae_encoder": device_vae_encoder.value,
    "vae_decoder": device_vae_decoder.value,
}

ov_pipe = OVWanImageToVideoPipeline(model_dir, device_map)

## Run OpenVINO Model Inference
[back to top ⬆️](#Table-of-contents:)

In [12]:
from diffusers.utils import export_to_video, load_image
import numpy as np

prompt = "Summer beach vacation style, a white cat wearing sunglasses sits on a surfboard. The fluffy-furred feline gazes directly at the camera with a relaxed expression. Blurred beach scenery forms the background featuring crystal-clear waters, distant green hills, and a blue sky dotted with white clouds. The cat assumes a naturally relaxed posture, as if savoring the sea breeze and warm sunlight. A close-up shot highlights the feline's intricate details and the refreshing atmosphere of the seaside."
negative_prompt = "Bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, static, overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, poorly drawn hands, poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, still picture, messy background, three legs, many people in the background, walking backwards"

image = load_image("https://raw.githubusercontent.com/Wan-Video/Wan2.2/main/examples/i2v_input.JPG")
max_area = 480 * 832
aspect_ratio = image.height / image.width
mod_value = ov_pipe.vae_scale_factor_spatial * ov_pipe.patch_size[1]
height = round(np.sqrt(max_area * aspect_ratio)) // mod_value * mod_value
width = round(np.sqrt(max_area / aspect_ratio)) // mod_value * mod_value
image = image.resize((width, height))

output = ov_pipe(
    image=image,
    prompt=prompt,
    negative_prompt=negative_prompt,
    height=height,
    width=width,
    num_frames=20,
    guidance_scale=5.0,
    num_inference_steps=4,
).frames[0]
export_to_video(output, "output_ov.mp4", fps=10)

  0%|          | 0/4 [00:00<?, ?it/s]

'output_ov.mp4'

In [13]:
from IPython.display import Video

display(Video("output_ov.mp4"))

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

In [ ]:
from gradio_helper import make_demo

demo = make_demo(ov_pipe)

try:
    demo.launch(debug=True)
except Exception:
    demo.launch(share=True, debug=True)
# if you are launching remotely, specify server_name and server_port
# demo.launch(server_name='your server name', server_port='server port in int')
# Read more in the docs: https://gradio.app/docs/